# Code LLM Forge — Complete Pipeline (Gemma 4 E4B)

> Fine-tune Gemma 4 E4B for code generation + tool calling, benchmark it, export to GGUF.

| Section | What It Does | Est. Time |
|---------|-------------|-----------|
| 0. Install | Dependencies (Unsloth official method) | 2 min |
| 1. Data | Download & format 50K code + 50K tool-calling samples | 15 min |
| 2. Train Stage 1 | Code + Reasoning SFT | ~1.5 hrs |
| 3. Train Stage 2 | Tool Calling SFT | ~1.5 hrs |
| 4. Benchmark | Custom zero-contamination eval | 30 min |
| 5. Export | GGUF for llama.cpp | 15 min |

**Total: ~4 hours on A100 80GB**

## 0. Install Dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install --no-deps --upgrade timm  # Needed for Gemma 4 model loading
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
import torch, os, json, gc, time, random
from datetime import datetime

# Reduce memory fragmentation — critical for large-vocab models like Gemma 4
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu} ({vram:.0f} GB)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

os.makedirs("/content/forge", exist_ok=True)
os.makedirs("/content/forge/stage1", exist_ok=True)
os.makedirs("/content/forge/stage2", exist_ok=True)
os.makedirs("/content/forge/benchmarks", exist_ok=True)
os.makedirs("/content/forge/gguf", exist_ok=True)
print("Directories ready")


## 1. Data Preparation

Download & format datasets:
- **Stage 1 (Code + Reasoning):** CodeFeedback + OpenOrca subset = 50K samples
- **Stage 2 (Tool Calling):** Glaive Function Calling = 50K samples

In [ ]:
from datasets import load_dataset

print("Downloading datasets...")

# Code instruction data
print("[1/3] CodeFeedback-Filtered-Instruction...")
ds_code = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train")
print(f"  -> {len(ds_code):,} samples")

# Reasoning data
print("[2/3] OpenOrca (30K sample)...")
ds_orca = load_dataset("Open-Orca/OpenOrca", split="train")
ds_orca = ds_orca.shuffle(seed=42).select(range(30000))
print(f"  -> {len(ds_orca):,} samples")

# Tool calling data
print("[3/3] Glaive Function Calling v2...")
ds_tools = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(f"  -> {len(ds_tools):,} samples")

print("All datasets loaded!")

In [ ]:
# Format into simple conversation lists
def fmt_code(ex):
    q = ex.get("query", ex.get("instruction", ""))
    a = ex.get("answer", ex.get("response", ""))
    if not q or not a: return None
    return [{"role":"user","content":q}, {"role":"assistant","content":a}]

def fmt_orca(ex):
    q = ex.get("question",""); a = ex.get("response","")
    if not q or not a: return None
    msgs = []
    s = ex.get("system_prompt","")
    if s: msgs.append({"role":"system","content":s})
    msgs += [{"role":"user","content":q}, {"role":"assistant","content":a}]
    return msgs

def fmt_tools(ex):
    chat = ex.get("chat","")
    if not chat: return None
    msgs = []
    sys_text = ex.get("system","")
    if sys_text:
        # Strip redundant 'SYSTEM: ' prefix since role=system handles it
        sys_clean = sys_text.replace("SYSTEM: ", "", 1).strip()
        if sys_clean: msgs.append({"role":"system","content":sys_clean})
    # Clean up <|endoftext|> markers from Glaive format
    chat = chat.replace("<|endoftext|>", "")
    for part in chat.split("USER: ")[1:]:
        if "ASSISTANT: " in part:
            u, a = part.split("ASSISTANT: ", 1)
            msgs.append({"role":"user","content":u.strip()})
            msgs.append({"role":"assistant","content":a.strip()})
    return msgs if len(msgs) >= 2 else None

# Process Stage 1
print("Formatting Stage 1 (Code + Reasoning)...")
stage1 = []
for ex in ds_code:
    r = fmt_code(ex)
    if r: stage1.append({"conversations": r})
for ex in ds_orca:
    r = fmt_orca(ex)
    if r: stage1.append({"conversations": r})
random.seed(42); random.shuffle(stage1)
stage1 = stage1[:50000]
print(f"  Stage 1: {len(stage1):,} samples")

# Process Stage 2
print("Formatting Stage 2 (Tool Calling)...")
stage2 = []
for ex in ds_tools:
    r = fmt_tools(ex)
    if r: stage2.append({"conversations": r})
random.shuffle(stage2)
stage2 = stage2[:50000]
print(f"  Stage 2: {len(stage2):,} samples")

# Save as JSONL
for name, data in [("stage1", stage1), ("stage2", stage2)]:
    path = f"/content/forge/{name}_data.jsonl"
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"  Saved {path}")

# Free memory
del ds_code, ds_orca, ds_tools
gc.collect()
print(f"\nTotal: {len(stage1)+len(stage2):,} samples ready")

## 2. Stage 1 — Code + Reasoning SFT

Using the **official Unsloth Gemma 4 E4B** pattern:
- `FastModel` (not FastLanguageModel)
- `unsloth/gemma-4-E4B-it` (optimized, KV cache bug fixed)
- `SFTConfig` (not TrainingArguments)
- `train_on_responses_only` (don't train on user prompts)
- LoRA rank 16, ~1.5 hrs on A100

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = 4096,  # 102GB VRAM — no need to compromise on seq length
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)
print("Model loaded!")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import Dataset

# Apply Gemma 4 chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Load and format Stage 1 data
with open("/content/forge/stage1_data.jsonl") as f:
    records = [json.loads(l) for l in f]
ds1 = Dataset.from_list(records)
ds1 = standardize_data_formats(ds1)
print(f"Stage 1 dataset: {len(ds1):,} samples")

# Apply chat template
def format_prompts(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        try:
            t = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            texts.append(t.removeprefix('<bos>'))
        except:
            texts.append("")
    return {"text": texts}

ds1 = ds1.map(format_prompts, batched=True, num_proc=4)  # parallel tokenization
ds1 = ds1.filter(lambda x: len(x["text"]) > 50)
print(f"After formatting: {len(ds1):,} samples")
print("Sample:", ds1[0]["text"][:300])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds1,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        # batch=8 + accum=2 = effective batch 16, but halves peak VRAM vs batch=16
        # Needed because 262k vocab causes logit spikes to 80GB+ at batch=16
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        warmup_steps = 10,
        logging_steps = 25,
        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/forge/stage1",
        report_to = "none",
        max_seq_length = 4096,
        bf16 = True,
        dataloader_num_workers = 4,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

print("Starting Stage 1 training...")
t0 = time.time()
stats1 = trainer.train()
t1 = time.time() - t0
print(f"\nStage 1 done! {t1/3600:.2f} hrs, loss={stats1.training_loss:.4f}")


In [ ]:
# Save merged Stage 1 model
print("Saving Stage 1 model...")
model.save_pretrained_merged("/content/forge/stage1/merged", tokenizer)
print("Stage 1 saved!")

del model, trainer, ds1
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e9:.1f} GB used")

## 3. Stage 2 — Tool Calling SFT

Fine-tune the Stage 1 model on function-calling data.
Lower learning rate (1e-4) to preserve code skills.

In [ ]:
# Load Stage 1 model
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = "/content/forge/stage1/merged",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)
print("Stage 2 model loaded with fresh LoRA adapters")


In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

with open("/content/forge/stage2_data.jsonl") as f:
    records = [json.loads(l) for l in f]
ds2 = Dataset.from_list(records)
ds2 = standardize_data_formats(ds2)

# Define format_prompts locally (same as Stage 1)
def format_prompts(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        try:
            t = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            texts.append(t.removeprefix('<bos>'))
        except:
            texts.append("")
    return {"text": texts}

ds2 = ds2.map(format_prompts, batched=True, num_proc=4)  # parallel tokenization
ds2 = ds2.filter(lambda x: len(x["text"]) > 50)
print(f"Stage 2 dataset: {len(ds2):,} samples")


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds2,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        warmup_steps = 10,
        logging_steps = 25,
        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/forge/stage2",
        report_to = "none",
        max_seq_length = 4096,
        bf16 = True,
        dataloader_num_workers = 4,
    ),
)

trainer2 = train_on_responses_only(
    trainer2,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

print("Starting Stage 2 training...")
t0 = time.time()
stats2 = trainer2.train()
t2 = time.time() - t0
print(f"\nStage 2 done! {t2/3600:.2f} hrs, loss={stats2.training_loss:.4f}")


In [ ]:
# Save final model
print("Saving final model...")
model.save_pretrained_merged("/content/forge/stage2/merged", tokenizer)
print("Final model saved!")

print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"Stage 1: {t1/3600:.2f} hrs, loss={stats1.training_loss:.4f}")
print(f"Stage 2: {t2/3600:.2f} hrs, loss={stats2.training_loss:.4f}")
print(f"Total: {(t1+t2)/3600:.2f} hrs")
print(f"{'='*50}")

del model, trainer2, ds2
gc.collect(); torch.cuda.empty_cache()

## 4. Benchmarks (Zero-Contamination)

Custom eval — these problems were never in any training set.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastModel.from_pretrained(
    model_name = "/content/forge/stage2/merged",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
print("Model loaded for benchmarking")

def ask(prompt, max_tokens=512):
    msgs = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", tokenize=True, return_dict=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=1.0, top_p=0.95, top_k=64, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Code Generation Tests
print("CODE GENERATION TESTS")
print("=" * 50)

code_tests = [
    ("Write a Python function `is_palindrome(s)` that checks if a string is a palindrome, ignoring case and non-alphanumeric characters.",
     lambda r: "def " in r and "palindrome" in r.lower()),
    ("Write a Python function `fibonacci(n)` that returns the nth Fibonacci number using dynamic programming.",
     lambda r: "def " in r and ("fib" in r.lower())),
    ("Write a Python function `merge_sorted(a, b)` that merges two sorted lists into one sorted list in O(n+m) time.",
     lambda r: "def " in r and "merge" in r.lower()),
]

code_pass = 0
for prompt, check in code_tests:
    resp = ask(prompt, max_tokens=512)
    passed = check(resp)
    code_pass += passed
    print(f"  {'PASS' if passed else 'FAIL'}: {prompt[:60]}...")

print(f"\nCode: {code_pass}/{len(code_tests)}")

# Tool Calling Tests
print("\nTOOL CALLING TESTS")
print("=" * 50)

tool_tests = [
    ("You have these tools: get_weather(city: str). User says: What's the weather in Tokyo? Call the function.",
     lambda r: "get_weather" in r and "tokyo" in r.lower()),
    ("You have these tools: send_email(to: str, subject: str, body: str). User says: What is 2+2? Only call a tool if needed.",
     lambda r: "4" in r and "send_email" not in r.lower()),
    ("You have these tools: search(query: str), calculate(expr: str). User says: What is the square root of 144? Call the right function.",
     lambda r: "calculate" in r or "12" in r),
]

tool_pass = 0
for prompt, check in tool_tests:
    resp = ask(prompt, max_tokens=256)
    passed = check(resp)
    tool_pass += passed
    print(f"  {'PASS' if passed else 'FAIL'}: {prompt[:60]}...")

print(f"\nTool Calling: {tool_pass}/{len(tool_tests)}")

# Reasoning Tests
print("\nREASONING TESTS")
print("=" * 50)

reason_tests = [
    ("You have 8 balls, one is heavier. What's the minimum weighings on a balance scale to find it? Explain step by step.",
     lambda r: "2" in r),
    ("A farmer has 100m of fencing for 3 sides of a rectangle (one side is a wall). What dimensions maximize area? Show your work.",
     lambda r: "50" in r or "25" in r or "1250" in r),
]

reason_pass = 0
for prompt, check in reason_tests:
    resp = ask(prompt, max_tokens=512)
    passed = check(resp)
    reason_pass += passed
    print(f"  {'PASS' if passed else 'FAIL'}: {prompt[:60]}...")

print(f"\nReasoning: {reason_pass}/{len(reason_tests)}")

# Summary
total_pass = code_pass + tool_pass + reason_pass
total_tests = len(code_tests) + len(tool_tests) + len(reason_tests)
print(f"\n{'='*50}")
print(f"OVERALL: {total_pass}/{total_tests} ({100*total_pass/total_tests:.0f}%)")
print(f"{'='*50}")

results = {"code": f"{code_pass}/{len(code_tests)}", "tools": f"{tool_pass}/{len(tool_tests)}", "reasoning": f"{reason_pass}/{len(reason_tests)}", "overall": f"{total_pass}/{total_tests}"}
with open("/content/forge/benchmarks/results.json","w") as f:
    json.dump(results, f, indent=2)
print("Results saved to /content/forge/benchmarks/results.json")

## 5. Export to GGUF

> **Note:** Gemma 4 currently only supports Q8_0, BF16, and F16 quantization in Unsloth.

In [ ]:
# Export to GGUF (Q8_0 — best quality Gemma 4 supports)
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = "/content/forge/stage2/merged",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False,  # Need full precision for GGUF
)

print("Exporting Q8_0 GGUF...")
model.save_pretrained_gguf(
    "/content/forge/gguf",
    tokenizer,
    quantization_method = "Q8_0",
)
print("\nGGUF export done!")

# List files
for f in sorted(os.listdir("/content/forge/gguf")):
    if f.endswith(".gguf"):
        sz = os.path.getsize(f"/content/forge/gguf/{f}") / 1e9
        print(f"  {f}: {sz:.2f} GB")

In [ ]:
# ============================================================
# DONE!
# ============================================================
print("=" * 50)
print("CODE LLM FORGE — COMPLETE!")
print("=" * 50)
try:
    print(f"Stage 1: {t1/3600:.2f} hrs")
    print(f"Stage 2: {t2/3600:.2f} hrs")
    print(f"Benchmark: {total_pass}/{total_tests}")
except NameError:
    print("(Some timing/benchmark variables not available)")
print(f"GGUF: /content/forge/gguf/")
print()
print("To download:")
print("  from google.colab import files")
print("  files.download('/content/forge/gguf/YOUR_FILE.gguf')")
print()
print("To run locally:")
print("  llama-cli -m model-Q8_0.gguf -co -cnv -fa -ngl 99")